# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

This dataset contains survey data on mental health indicators among residents of Kilifi County, including demographic details and psychological symptom scores (GAD-7, PHQ-9, PSQ).

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}\n")
print(f"Published: {meta.datePublished}, Version: {meta.version}")
print(f"License: {meta.license}")
print(f"Keywords: {', '.join(meta.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate all record sets, list their `@id`, name and associated fields with their `@id`s. All references are by their Croissant `@id`.

In [ ]:
def print_records_overview(dataset):
    print("Available record sets in this dataset:")
    record_sets = dataset.metadata.record_sets
    for rs in record_sets:
        print(f"\nRecord set: [@id] {rs['@id']}, Name: {rs.get('name', '<no name>')}")
        fields = rs.get('fields', [])
        if not fields:
            print("    No explicit fields listed.")
        else:
            print("    Fields:")
            for f in fields:
                if isinstance(f, dict):
                    print(f"        [@id] {f.get('@id','')} - {f.get('name','')}")
                else:
                    print(f"        [@id] {f}")
    print()

print_records_overview(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we extract all tabular record sets and list sample fields. All IDs are used as per the Croissant standard.

In [ ]:
# Get all record set IDs (by @id) defined in metadata
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in record_sets:
    # Load all records for this record set
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from [@id]={record_set_id}. Columns:")
            print(df.columns.tolist())
            print(df.head(2))
        else:
            print(f"No records available for record set [@id]={record_set_id}.")
    except Exception as e:
        print(f"Could not load record set [@id]={record_set_id} due to: {e}")

# Choose a record set for continued analysis: take the first tabular record set
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    print(f"\nUsing record set [@id]={selected_record_set_id} for detailed analysis.")
else:
    selected_record_set_id = None
    df = None

## 4. Exploratory Data Analysis (EDA)
Apply common processing: filtering, normalizing numeric fields, grouping, and basic summary statistics.

All operations refer to fields/columns by their Croissant `@id`.

In [ ]:
if df is not None:
    # Attempt to identify a numeric field using column meta-data or names
    numeric_candidates = [c for c in df.columns if df[c].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_candidates:
        # Fallback: heuristic match for common numeric fields
        for col in df.columns:
            if 'score' in col.lower() or 'age' in col.lower() or 'gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower():
                numeric_candidates.append(col)
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Use the first available
        print(f"Using numeric field [@id]={numeric_field}\n")
        threshold = df[numeric_field].mean()  # Use mean as threshold example

        # Filter records with numeric_field > threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (count={len(filtered_df)}):\n", filtered_df.head())

        # Normalize field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt group by a categorical column (e.g., gender, village, etc.)
        possible_groups = [c for c in df.columns if c != numeric_field and pd.api.types.is_string_dtype(df[c])]
        if possible_groups:
            group_field = possible_groups[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nMean of {numeric_field} grouped by {group_field}:")
            print(grouped_df)
        else:
            print("No categorical field available for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if df is not None and 'numeric_field' in locals():
    # Histogram of the selected numeric field
    plt.figure(figsize=(8,4))
    df[numeric_field].dropna().hist(bins=20)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If a group field was identified, show boxplot
    if 'group_field' in locals():
        df.boxplot(column=numeric_field, by=group_field, figsize=(10,5))
        plt.title(f'{numeric_field} by {group_field}')
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion
This notebook demonstrated the use of the `mlcroissant` library to access and explore a Croissant-compliant dataset. By referencing all entities (record sets, fields, columns) strictly by their `@id`, we ensured schema compliance and reproducibility. Further, we loaded the data for analysis and demonstrated some typical exploratory steps, including filtering, normalization, grouping, and plotting data distributions.

For a more extensive analysis, one could further investigate relationships between psychological scores and demographic factors, predict risk, or explore regional trends. Refer to the dataset documentation for detailed field definitions.